In [1]:
# ==================== ViT 模型训练（Jupyter 单单元格版本，完整优化版）====================
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import os
import time
import gc
from PIL import Image
from torchvision import transforms
from tqdm import tqdm
from torch.utils.data import DataLoader, WeightedRandomSampler
from torch.utils.tensorboard import SummaryWriter
from transformers import ViTForImageClassification, ViTImageProcessor
from sklearn.metrics import precision_score, recall_score, f1_score
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

import sys
sys.path.append('..')

try:
    from src.data.config import config, device
    from src.data.dataset import FundusDataset
    from src.data.class_balance import calculate_multilabel_weights, create_multilabel_balanced_sampler, WeightedBCEWithLogitsLoss
    from src.data.focal_loss import FocalLoss
    print("✅ 自定义模块导入成功！")
except Exception as e:
    print(f"❌ 自定义模块导入失败: {e}")
    raise

# ========== 组合损失函数 ==========
class CombinedLoss(nn.Module):
    def __init__(self, class_weights, alpha=0.5, gamma=2.0):
        super().__init__()
        self.bce = WeightedBCEWithLogitsLoss(class_weights=class_weights, reduction='mean')
        self.focal = FocalLoss(alpha=class_weights, gamma=gamma, reduction='mean')
        self.alpha = alpha
    
    def forward(self, inputs, targets):
        return self.alpha * self.bce(inputs, targets) + (1 - self.alpha) * self.focal(inputs, targets)

# ========== 主训练函数 ==========
def main():
    print("\n" + "="*60)
    print("ViT 模型训练开始（完整优化版）")
    print("="*60 + "\n")
    
    # 1. 数据路径设置
    train_dir = r'C:\Users\lenovo\Desktop\graduation_project\data\latest\Training Images'
    test_dir = r'C:\Users\lenovo\Desktop\graduation_project\data\latest\Testing Images'
    val_dir = r'C:\Users\lenovo\Desktop\graduation_project\data\latest\validation images'
    excel_dir = r'C:\Users\lenovo\Desktop\graduation_project\data\raw\ODIR-5K\data.xlsx'
    
    print("📁 数据路径设置完成")
    
    # 2. 加载数据集
    print("\n📊 加载数据集...")
    train_dataset = FundusDataset(train_dir, excel_dir, is_training=True)
    val_dataset = FundusDataset(val_dir, excel_dir, is_training=False)
    test_dataset = FundusDataset(test_dir, excel_dir, is_training=False)
    
    print(f"训练集大小: {len(train_dataset)}")
    print(f"验证集大小: {len(val_dataset)}")
    print(f"测试集大小: {len(test_dataset)}")
    
    # 3. 收集训练集标签
    print("\n⚖️ 收集训练集标签...")
    all_train_labels = []
    for i in tqdm(range(len(train_dataset)), desc="收集标签"):
        _, labels, _ = train_dataset[i]
        all_train_labels.append(labels.numpy())
        if i % 1000 == 0:
            gc.collect()
    
    all_train_labels = np.stack(all_train_labels)
    print(f"标签数组形状: {all_train_labels.shape}")
    
    # 计算基础类别权重
    class_weights = calculate_multilabel_weights(
        all_train_labels,
        beta=0.999,
        clip_min=0.5,
        clip_max=5.0
    )
    print(f"\n基础类别权重:")
    for i, w in enumerate(class_weights):
        print(f"  类别 {i}: {w:.3f}")
    
    # ========== 极端过采样策略 ==========
    print("\n🔄 创建极端过采样策略...")
    
    # 根据样本量定义不同级别的少数类
    extreme_minority = [4, 5]      # 样本最少 (232, 146)
    medium_minority = [2, 3, 6]    # 中等少数 (304, 296, 246)
    
    sample_weights = []
    for i in range(len(train_dataset)):
        _, labels, _ = train_dataset[i]
        
        # 极端少数类权重 20x
        if labels[5] == 1 or labels[4] == 1:
            weight = 20.0
        # 中等少数类权重 8x
        elif any(labels[j] == 1 for j in medium_minority):
            weight = 8.0
        # 多数类权重 1x
        else:
            weight = 1.0
        
        sample_weights.append(weight)
    
    print(f"权重分布: 极端少数类(4,5)=20x, 中等少数类(2,3,6)=8x, 多数类=1x")
    print(f"样本权重范围: min={min(sample_weights):.1f}, max={max(sample_weights):.1f}")
    
    # 创建过采样器
    oversampler = WeightedRandomSampler(
        weights=torch.tensor(sample_weights, dtype=torch.double),
        num_samples=len(sample_weights) * 3,
        replacement=True
    )
    
    # 4. 创建 DataLoader
    print("\n🔄 创建 DataLoader...")
    batch_size = config.get('batch_size', 16)
    
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        sampler=oversampler,
        num_workers=0,
        pin_memory=False,
        drop_last=True
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=False
    )
    
    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=False
    )
    
    print(f"训练批次: {len(train_loader)}")
    print(f"验证批次: {len(val_loader)}")
    
    # 5. 加载预训练模型
    print("\n🤖 加载预训练模型...")
    local_model_path = config['pretrained_path']
    
    processor = ViTImageProcessor.from_pretrained(local_model_path)
    print("✅ 处理器加载成功")
    
    model = ViTForImageClassification.from_pretrained(
        local_model_path,
        num_labels=config['num_classes'],
        ignore_mismatched_sizes=True
    )
    model = model.to(device)
    print(f"✅ 模型加载成功！参数量: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")
    
    # 6. 配置损失函数（使用调整后的权重）
    print("\n⚙️ 配置损失函数和优化器...")
    
    # 调整类别权重，重点关注极少数类
    adjusted_weights = class_weights.clone()
    adjusted_weights[4] = 5.0   # 类别4权重提高到5.0
    adjusted_weights[5] = 8.0   # 类别5权重提高到8.0
    adjusted_weights = adjusted_weights.to(device)
    
    print(f"调整后类别权重:")
    for i, w in enumerate(adjusted_weights.cpu().numpy()):
        print(f"  类别 {i}: {w:.3f}")
    
    # 使用组合损失
    criterion = CombinedLoss(
        class_weights=adjusted_weights,
        alpha=0.5,
        gamma=2.0
    )
    print("✅ 使用组合损失: 0.5*BCE + 0.5*Focal, gamma=2.0")
    
    # 优化器 - 较高初始学习率
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=3e-5,
        weight_decay=config.get('weight_decay', 0.01)
    )
    
    # 余弦退火调度器
    scheduler = CosineAnnealingWarmRestarts(
        optimizer,
        T_0=5,
        T_mult=2,
        eta_min=1e-6
    )
    
    # 7. 初始化训练变量
    best_val_f1 = 0
    patience_counter = 0
    early_stop_patience = 20
    epochs = 40  # 增加到40轮
    global_step = 0
    phase1_epochs = 15  # 前15轮为阶段1
    
    os.makedirs(config['save_dir'], exist_ok=True)
    os.makedirs('../logs', exist_ok=True)
    
    writer = SummaryWriter('../logs')
    print(f"✅ TensorBoard 日志保存到: ../logs")
    
    print(f"\n🚀 开始训练，共 {epochs} 轮")
    print(f"阶段1 (前{phase1_epochs}轮): 学习率 3e-5")
    print(f"阶段2 (后{epochs-phase1_epochs}轮): 学习率 1e-5")
    print("="*60)
    
    train_start_time = time.time()
    
    # 8. 训练循环
    for epoch in range(epochs):
        # 阶段切换
        if epoch == phase1_epochs:
            print("\n🔄 切换到阶段2: 降低学习率，聚焦少数类")
            for param_group in optimizer.param_groups:
                param_group['lr'] = 1e-5
        
        model.train()
        train_loss = 0
        train_correct = 0
        train_total = 0
        train_strict_correct = 0
        train_strict_total = 0
        
        train_process = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs} [Train]')
        
        for image, labels, img_name in train_process:
            image = image.to(device)
            labels = labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(image)
            loss = criterion(outputs.logits, labels)
            
            probability = torch.sigmoid(outputs.logits)
            predict = (probability > 0.5).float()
            
            strict_correct = (predict == labels).all(dim=1).sum().item()
            strict_total = labels.size(0)
            strict_acc = strict_correct / strict_total if strict_total > 0 else 0
            
            total_correct = (predict == labels).sum().item()
            total_elements = labels.numel()
            batch_acc = total_correct / total_elements if total_elements > 0 else 0
            
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            train_strict_correct += strict_correct
            train_strict_total += strict_total
            train_correct += total_correct
            train_total += total_elements
            
            writer.add_scalar('Batch/Train_Loss', loss.item(), global_step)
            writer.add_scalar('Batch/Train_Strict_Accuracy', strict_acc, global_step)
            writer.add_scalar('Batch/Train_Label_Accuracy', batch_acc, global_step)
            global_step += 1
            
            train_process.set_postfix({
                'loss': f'{loss.item():.4f}',
                'acc': f'{batch_acc:.3f}',
                'strict': f'{strict_acc:.3f}'
            })
        
        # 计算epoch平均值
        train_strict_accuracy = train_strict_correct / train_strict_total if train_strict_total > 0 else 0
        train_label_accuracy = train_correct / train_total if train_total > 0 else 0
        avg_train_loss = train_loss / len(train_loader)
        
        writer.add_scalar('Loss/Train', avg_train_loss, epoch)
        writer.add_scalar('Accuracy/Train_Strict', train_strict_accuracy, epoch)
        writer.add_scalar('Accuracy/Train_Label', train_label_accuracy, epoch)
        current_lr = optimizer.param_groups[0]['lr']
        writer.add_scalar('Hyperparameters/Learning_Rate', current_lr, epoch)
        
        print(f"\nEpoch {epoch + 1}:")
        print(f"  损失: {avg_train_loss:.4f}")
        print(f"  标签准确率: {train_label_accuracy:.3f}")
        print(f"  严格准确率: {train_strict_accuracy:.3f}")
        
        # 验证阶段
        model.eval()
        all_val_probs = []
        all_val_labels = []
        
        with torch.no_grad():
            for val_images, val_labels, _ in tqdm(val_loader, desc=f'Epoch {epoch+1} [Val]'):
                val_images = val_images.to(device)
                val_outputs = model(val_images)
                val_probs = torch.sigmoid(val_outputs.logits).cpu().numpy()
                all_val_probs.append(val_probs)
                all_val_labels.append(val_labels.numpy())
        
        if all_val_probs:
            all_val_probs = np.vstack(all_val_probs)
            all_val_labels = np.vstack(all_val_labels)
            
            # 为每个类别寻找最佳阈值
            best_thresholds = []
            per_class_best_f1 = []
            
            for i in range(8):
                best_f1 = 0
                best_th = 0.5
                for th in np.arange(0.2, 0.9, 0.05):
                    preds = (all_val_probs[:, i] > th).astype(int)
                    f1 = f1_score(all_val_labels[:, i], preds, zero_division=0)
                    if f1 > best_f1:
                        best_f1 = f1
                        best_th = th
                best_thresholds.append(best_th)
                per_class_best_f1.append(best_f1)
            
            # 使用最佳阈值
            val_preds = np.zeros_like(all_val_probs)
            for i in range(8):
                val_preds[:, i] = (all_val_probs[:, i] > best_thresholds[i]).astype(int)
            
            val_f1 = f1_score(all_val_labels, val_preds, average='macro', zero_division=0)
            per_class_f1 = f1_score(all_val_labels, val_preds, average=None, zero_division=0)
            
            print(f"  最佳阈值: {[f'{th:.2f}' for th in best_thresholds]}")
            print(f"  各类别F1: {[f'{f1:.3f}' for f1 in per_class_f1]}")
            
            # 特别关注极少数类
            extreme_minority_f1 = np.mean([per_class_f1[4], per_class_f1[5]])
            minority_f1 = np.mean([per_class_f1[2], per_class_f1[3], per_class_f1[6]])
            print(f"  极少数类(4,5)平均F1: {extreme_minority_f1:.4f}")
            print(f"  中等少数类(2,3,6)平均F1: {minority_f1:.4f}")
            print(f"  验证集Macro F1: {val_f1:.4f}")
            
            writer.add_scalar('Val/Macro_F1', val_f1, epoch)
            writer.add_scalar('Val/Extreme_Minority_F1', extreme_minority_f1, epoch)
            writer.add_scalar('Val/Minority_F1', minority_f1, epoch)
            
            if val_f1 > best_val_f1:
                best_val_f1 = val_f1
                patience_counter = 0
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'loss': avg_train_loss,
                    'strict_accuracy': train_strict_accuracy,
                    'label_accuracy': train_label_accuracy,
                    'val_f1': val_f1,
                    'per_class_f1': per_class_f1,
                    'best_thresholds': best_thresholds,
                    'config': config
                }, os.path.join(config['save_dir'], 'best_model_by_val_f1.pth'))
                print(f"  ✅ 保存验证集最佳模型！F1={val_f1:.4f}")
            else:
                patience_counter += 1
            
            scheduler.step()
        
        # 保存最新模型
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': avg_train_loss,
            'accuracy': train_strict_accuracy,
            'config': config
        }, os.path.join(config['save_dir'], 'latest_model.pth'))
        
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        
        print("-" * 50)
        
        if patience_counter >= early_stop_patience:
            print(f"\n⏹️ 早停: {early_stop_patience}个epoch F1未提升")
            break
    
    # 训练总结
    train_time = time.time() - train_start_time
    print("\n" + "="*60)
    print("🏁 训练完成！")
    print("="*60)
    print(f"总训练时间: {train_time / 60:.2f} 分钟")
    print(f"最佳验证集F1: {best_val_f1:.4f}")
    print(f"模型保存位置: {config['save_dir']}")
    print(f"TensorBoard日志位置: ../logs")
    
    writer.close()
    return model, best_val_f1

# ========== 运行 ==========
print("开始训练...")
model, best_f1 = main()
print(f"训练结束，最佳F1: {best_f1:.4f}")

%load_ext tensorboard
%tensorboard --logdir ../logs

✅ 自定义模块导入成功！
开始训练...

ViT 模型训练开始（完整优化版）

📁 数据路径设置完成

📊 加载数据集...
训练集大小: 4906
验证集大小: 1046
测试集大小: 1048

⚖️ 收集训练集标签...


收集标签: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 4906/4906 [00:41<00:00, 119.07it/s]


标签数组形状: (4906, 8)

📊 多标签类别统计:
  类别 0: 正样本=1596, 负样本=3310, 正比例=0.325
  类别 1: 正样本=1584, 负样本=3322, 正比例=0.323
  类别 2: 正样本=304, 负样本=4602, 正比例=0.062
  类别 3: 正样本=296, 负样本=4610, 正比例=0.060
  类别 4: 正样本=232, 负样本=4674, 正比例=0.047
  类别 5: 正样本=146, 负样本=4760, 正比例=0.030
  类别 6: 正样本=246, 负样本=4660, 正比例=0.050
  类别 7: 正样本=1374, 负样本=3532, 正比例=0.280

📊 类别权重:
  类别 0: 0.500
  类别 1: 0.500
  类别 2: 1.077
  类别 3: 1.102
  类别 4: 1.363
  类别 5: 2.078
  类别 6: 1.294
  类别 7: 0.500

基础类别权重:
  类别 0: 0.500
  类别 1: 0.500
  类别 2: 1.077
  类别 3: 1.102
  类别 4: 1.363
  类别 5: 2.078
  类别 6: 1.294
  类别 7: 0.500

🔄 创建极端过采样策略...


Some weights of ViTForImageClassification were not initialized from the model checkpoint at C:/Users/lenovo/Desktop/graduation_project/models/vit-base-patch16-224 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([8]) in the model instantiated
- classifier.weight: found shape torch.Size([1000, 768]) in the checkpoint and torch.Size([8, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


权重分布: 极端少数类(4,5)=20x, 中等少数类(2,3,6)=8x, 多数类=1x
样本权重范围: min=1.0, max=20.0

🔄 创建 DataLoader...
训练批次: 919
验证批次: 66

🤖 加载预训练模型...
✅ 处理器加载成功
✅ 模型加载成功！参数量: 85.80M

⚙️ 配置损失函数和优化器...
调整后类别权重:
  类别 0: 0.500
  类别 1: 0.500
  类别 2: 1.077
  类别 3: 1.102
  类别 4: 5.000
  类别 5: 8.000
  类别 6: 1.294
  类别 7: 0.500
✅ 使用组合损失: 0.5*BCE + 0.5*Focal, gamma=2.0
✅ TensorBoard 日志保存到: ../logs

🚀 开始训练，共 40 轮
阶段1 (前15轮): 学习率 3e-5
阶段2 (后25轮): 学习率 1e-5


Epoch 1/40 [Train]:   7%|█████▌                                                                             | 62/919 [00:53<12:24,  1.15it/s, loss=0.5024, acc=0.797, strict=0.062]


KeyboardInterrupt: 

In [6]:
"""
无需命令行的类别分布分析脚本
直接在 IPython/Jupyter 中运行
"""

import numpy as np

def check_class_distribution(y_train, y_val, class_names=None):
    """
    分析训练集和验证集的类别分布
    """
    # 判断标签格式
    if len(y_train.shape) > 1 and y_train.shape[1] > 1:
        # one-hot 编码
        train_dist = np.mean(y_train, axis=0)
        val_dist = np.mean(y_val, axis=0)
        train_counts = np.sum(y_train, axis=0)
        val_counts = np.sum(y_val, axis=0)
        n_classes = y_train.shape[1]
    else:
        # 整数编码
        n_classes = len(np.unique(np.concatenate([y_train, y_val])))
        train_dist = np.array([np.mean(y_train == i) for i in range(n_classes)])
        val_dist = np.array([np.mean(y_val == i) for i in range(n_classes)])
        train_counts = np.array([np.sum(y_train == i) for i in range(n_classes)])
        val_counts = np.array([np.sum(y_val == i) for i in range(n_classes)])
    
    if class_names is None:
        class_names = [f"类别{i}" for i in range(n_classes)]
    
    print("=" * 60)
    print("类别分布分析报告")
    print("=" * 60)
    
    print("\n📊 各类别在训练集中的比例:")
    print("-" * 40)
    for i, name in enumerate(class_names):
        print(f"  {name}: {train_dist[i]:.3f} ({int(train_counts[i])} 样本)")
    
    print("\n📊 各类别在验证集中的比例:")
    print("-" * 40)
    for i, name in enumerate(class_names):
        print(f"  {name}: {val_dist[i]:.3f} ({int(val_counts[i])} 样本)")
    
    # 计算分布差异
    diff = np.abs(train_dist - val_dist)
    print("\n📈 分布差异:")
    print("-" * 40)
    for i, name in enumerate(class_names):
        print(f"  {name}: {diff[i]:.3f}")
    
    # 统计信息
    print("\n📉 统计摘要:")
    print("-" * 40)
    print(f"  训练集总样本数: {int(np.sum(train_counts))}")
    print(f"  验证集总样本数: {int(np.sum(val_counts))}")
    print(f"  类别数量: {n_classes}")
    print(f"  平均分布差异: {np.mean(diff):.3f}")
    
    # 检查不平衡
    imbalance_ratio = np.max(train_counts) / np.min(train_counts)
    print(f"\n⚠️  不平衡比率: {imbalance_ratio:.2f}")
    if imbalance_ratio > 2:
        print("  警告: 数据集存在明显不平衡！")
    
    return {
        'train_dist': train_dist,
        'val_dist': val_dist,
        'diff': diff,
        'train_counts': train_counts,
        'val_counts': val_counts,
        'imbalance_ratio': imbalance_ratio
    }

def generate_sample_data():
    """生成示例数据"""
    np.random.seed(42)
    n_classes = 5
    n_samples = 1000
    
    # 生成不平衡数据
    train_probs = np.array([0.4, 0.25, 0.15, 0.12, 0.08])
    y_train = np.random.choice(n_classes, size=n_samples, p=train_probs)
    y_val = np.random.choice(n_classes, size=n_samples//5, p=train_probs)
    
    # 转换为 one-hot
    y_train_onehot = np.eye(n_classes)[y_train]
    y_val_onehot = np.eye(n_classes)[y_val]
    
    return y_train_onehot, y_val_onehot, ['A', 'B', 'C', 'D', 'E']

# 在 IPython/Jupyter 中直接运行这部分
if __name__ == "__main__":
    print("运行示例数据...")
    y_train, y_val, class_names = generate_sample_data()
    check_class_distribution(y_train, y_val, class_names)

运行示例数据...
类别分布分析报告

📊 各类别在训练集中的比例:
----------------------------------------
  A: 0.421 (421 样本)
  B: 0.250 (250 样本)
  C: 0.130 (130 样本)
  D: 0.120 (120 样本)
  E: 0.079 (79 样本)

📊 各类别在验证集中的比例:
----------------------------------------
  A: 0.360 (72 样本)
  B: 0.205 (41 样本)
  C: 0.165 (33 样本)
  D: 0.150 (30 样本)
  E: 0.120 (24 样本)

📈 分布差异:
----------------------------------------
  A: 0.061
  B: 0.045
  C: 0.035
  D: 0.030
  E: 0.041

📉 统计摘要:
----------------------------------------
  训练集总样本数: 1000
  验证集总样本数: 200
  类别数量: 5
  平均分布差异: 0.042

⚠️  不平衡比率: 5.33
  警告: 数据集存在明显不平衡！


In [2]:
# 在加载数据集后添加
val_labels_list = []
for i in range(len(val_dataset)):
    _, labels, _ = val_dataset[i]
    val_labels_list.append(labels.numpy())
val_labels = np.stack(val_labels_list)

print("验证集各类别正样本数:", val_labels.sum(axis=0))
print("训练集各类别正样本数:", all_train_labels.sum(axis=0))

# 计算分布差异
for i in range(8):
    train_ratio = all_train_labels[:, i].mean()
    val_ratio = val_labels[:, i].mean()
    print(f"类别 {i}: 训练集比例={train_ratio:.3f}, 验证集比例={val_ratio:.3f}, 差异={abs(train_ratio-val_ratio):.3f}")

NameError: name 'val_dataset' is not defined

In [ ]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import os
import time
from PIL import Image
from torchvision import transforms
from tqdm.notebook import tqdm
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from src.data.dataset import FundusDataset
from transformers import ViTForImageClassification, ViTImageProcessor
from sklearn.metrics import precision_score, recall_score, f1_score
from torch.utils.data import WeightedRandomSampler
# 在现有导入后面添加这一行
from src.data.class_balance import WeightedBCEWithLogitsLoss
from sklearn.utils.class_weight import compute_class_weight
# 超参数
config = {
    'model_name': 'google/vit-base-patch16-224',  # 预训练ViT模型
    'num_classes': 8,                              # 分类数
    'class_names': ['正常', '糖尿病视网膜病变', '青光眼', '白内障', 
                   '黄斑变性', '高血压视网膜病变', '近视', '其他'],
    'batch_size': 16,                               # 批次大小(16)
    'epochs': 20,                                   # 训练轮数(30)
    'learning_rate': 1e-5,                          # 学习率
     'weight_decay': 0.05,                           # 权重衰减
     'warmup_steps': 500,                            # 预热步数
     'gradient_accumulation_steps': 2,               # 梯度累积
    'max_grad_norm': 1.0,                           # 梯度裁剪
     'save_dir': './checkpoints',                    # 模型保存目录
    'log_dir': './logs',                              # TensorBoard日志目录
    'model_path': 'checkpoints/best_model.pth'       #模型位置
 }

def cpu():
    if torch.cuda.is_available():
        return 'cuda'
    else:
        return 'cpu'

result = cpu()
print(result)
device = torch.device(result)
print(device)

train_dir = r'C:\Users\lenovo\Desktop\graduation_project\data\latest\Training Images'
test_dir = r'C:\Users\lenovo\Desktop\graduation_project\data\latest\Testing Images'
val_dir = r'C:\Users\lenovo\Desktop\graduation_project\data\latest\validation Images'
excel_dir = r'C:\Users\lenovo\Desktop\graduation_project\data\raw\ODIR-5K\data.xlsx'

# 训练集
train_dataset = FundusDataset(train_dir ,excel_dir , is_training=True)

# 验证集
val_dataset = FundusDataset(val_dir ,excel_dir , is_training=False)

# 测试集
test_dataset = FundusDataset(test_dir ,excel_dir , is_training=False)

# 创建DataLoader
train_loader = DataLoader(
    train_dataset,
    batch_size=config['batch_size'],
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    drop_last=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config['batch_size'],
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=config['batch_size'],
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

# 用已经预训练过的模型可以增加准确率
local_model_path = r'C:/Users/lenovo/Desktop/graduation_project/models/vit-base-patch16-224'


# 加载预训练的处理器
processor = ViTImageProcessor.from_pretrained(local_model_path)

# 加载预训练的ViT模型
model = ViTForImageClassification.from_pretrained(
    local_model_path,
    num_labels=config['num_classes'],  # 指定分类数
    ignore_mismatched_sizes=True  # 允许模型权重尺寸不匹配时自动调整
)
model = model.to(device)

# 回归任务(预测连续值)：MSELoss,L1Loss,SmoothL1Loss
# 分类任务：1.多类分类(每个样本只属一类)：CrossEntropyLoss   2.多标签分类(每个样本可属多类)：BCEWithLogitsLoss


#数值更稳定，每个类别可独立判断
pos_rates = [0.06, 0.08, 0.02, 0.04, 0.04, 0.10, 0.06, 0.72]
pos_weight = torch.tensor([1.0/(r+0.01) for r in pos_rates]).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight) # 损失函数

# 优化器
optimizer = torch.optim.AdamW(  #AdamW收敛快，准确率高，泛化性好
    model.parameters(),
    lr=config['learning_rate'],
    weight_decay=config['weight_decay']
)

# 训练阶段
best_val = 0
global_step = 0
best_accuracy = 0
best_val_f1 = 0  # 记录最好的验证集F1

writer = SummaryWriter('logs')

# 循环训练
for epoch in range(config['epochs']):
    model.train()
    train_loss = 0
    train_correct = 0      # 改为累计正确标签数
    train_total = 0         # 改为累计总标签数
    train_strict_correct = 0  # 累计严格正确样本数
    train_strict_total = 0    # 累计总样本数
    
    train_process = tqdm(train_loader, desc='Training')
    
    for image, labels, img_name in train_process:
        
        image = image.to(device)
        labels = labels.to(device)
    
        optimizer.zero_grad()
        outputs = model(image)
        loss = criterion(outputs.logits, labels)

        probability = torch.sigmoid(outputs.logits)
        predict = (probability > 0.5).float()

        if epoch == 0 and train_process.n == 0:
            print(f"\n=== 调试信息 ===")
            print(f"batch_size: {labels.size(0)}")
            print(f"预测值范围: {probability.min():.3f} - {probability.max():.3f}")
            print(f"预测值均值: {probability.mean():.3f}")
            print(f"真实标签均值: {labels.mean():.3f}")
            print(f"预测正确率: {(predict == labels).float().mean():.3f}")
            print(f"预测样本: \n{predict[:3]}")
            print(f"真实样本: \n{labels[:3]}")
        
        # 1. 严格准确率（8个标签全对）- 用于保存最佳模型
        strict_correct = (predict == labels).all(dim=1).sum().item()
        strict_total = labels.size(0)
        strict_acc = strict_correct / strict_total
        
        # 2. 宽松准确率（每个标签独立计算）- 用于显示训练进度
        total_correct = (predict == labels).sum().item()
        total_elements = labels.numel()  # batch_size * 8
        batch_acc = total_correct / total_elements
        
        loss.backward()
        optimizer.step()

        # 累计损失
        train_loss += loss.item()
        
        # 累计两种准确率的计数
        train_strict_correct += strict_correct
        train_strict_total += strict_total
        train_correct += total_correct
        train_total += total_elements
        
        # 在进度条中显示宽松准确率
        train_process.set_postfix({
            'loss': f'{loss.item():.4f}',
            'acc': f'{batch_acc:.3f}',
            'strict': f'{strict_acc:.3f}'
        })

        writer.add_scalar('Batch/Train_Loss', loss.item(), global_step)
        writer.add_scalar('Batch/Train_Strict_Accuracy', strict_acc, global_step)
        writer.add_scalar('Batch/Train_Label_Accuracy', batch_acc, global_step)
        global_step += 1
    
    # ========== 计算epoch平均值 ==========
    # 严格准确率（用于保存最佳模型）
    train_strict_accuracy = train_strict_correct / train_strict_total
    # 宽松准确率（每个标签的准确率）
    train_label_accuracy = train_correct / train_total
    # 平均损失
    avg_train_loss = train_loss / len(train_loader)

    # 记录到TensorBoard
    writer.add_scalar('Loss/Train', avg_train_loss, epoch)
    writer.add_scalar('Accuracy/Train_Strict', train_strict_accuracy, epoch)
    writer.add_scalar('Accuracy/Train_Label', train_label_accuracy, epoch)
    current_lr = optimizer.param_groups[0]['lr']
    writer.add_scalar('Hyperparameters/Learning_Rate', current_lr, epoch)

    print(f"\nEpoch {epoch+1}:")
    print(f"  损失: {avg_train_loss:.4f}")
    print(f"  标签准确率: {train_label_accuracy:.3f} (每个标签)")
    print(f"  严格准确率: {train_strict_accuracy:.3f} (8标签全对)")

    # ========== 验证集F1计算 ==========
    model.eval()
    all_val_probs = []
    all_val_labels = []
    
    with torch.no_grad():
        for val_images, val_labels, _ in val_loader:
            val_images = val_images.to(device)
            val_outputs = model(val_images)
            val_probs = torch.sigmoid(val_outputs.logits).cpu().numpy()
            
            all_val_probs.append(val_probs)
            all_val_labels.append(val_labels.numpy())
    
    all_val_probs = np.vstack(all_val_probs)
    all_val_labels = np.vstack(all_val_labels)
    
    # 计算验证集F1
    val_preds = (all_val_probs > 0.5).astype(int)
    val_f1 = f1_score(all_val_labels, val_preds, average='macro', zero_division=0)
    
    print(f"  验证集Macro F1: {val_f1:.4f}")
    
    # 用验证集F1保存模型
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': avg_train_loss,
            'strict_accuracy': train_strict_accuracy,
            'label_accuracy': train_label_accuracy,
            'val_f1': val_f1,
            'config': config
        }, os.path.join(config['save_dir'], 'best_model_by_val_f1.pth'))
        print(f"  保存验证集最佳模型！F1={val_f1:.4f}")

    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': avg_train_loss,
        'accuracy': train_strict_accuracy,
        'config': config
    }, os.path.join(config['save_dir'], 'latest_model_2.pth'))
    
    if train_strict_accuracy > best_accuracy:
        best_accuracy = train_strict_accuracy
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': avg_train_loss,
            
            'accuracy': train_strict_accuracy,
            'config': config
        }, os.path.join(config['save_dir'], 'best_model_2.pth'))

writer.close()

cuda
cuda


Some weights of ViTForImageClassification were not initialized from the model checkpoint at C:/Users/lenovo/Desktop/graduation_project/models/vit-base-patch16-224 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([8]) in the model instantiated
- classifier.weight: found shape torch.Size([1000, 768]) in the checkpoint and torch.Size([8, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Training:   0%|          | 0/306 [00:00<?, ?it/s]


=== 调试信息 ===
batch_size: 16
预测值范围: 0.257 - 0.796
预测值均值: 0.535
真实标签均值: 0.133
预测正确率: 0.375
预测样本: 
tensor([[0., 0., 1., 0., 0., 1., 0., 1.],
        [1., 0., 0., 0., 1., 1., 1., 1.],
        [1., 0., 0., 0., 1., 1., 0., 1.]], device='cuda:0')
真实样本: 
tensor([[1., 0., 0., 0., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0., 0., 0., 1.]], device='cuda:0')

Epoch 1:
  损失: 1.0036
  标签准确率: 0.679 (每个标签)
  严格准确率: 0.000 (8标签全对)
  验证集Macro F1: 0.3006
  保存验证集最佳模型！F1=0.3006


Training:   0%|          | 0/306 [00:00<?, ?it/s]

...

Training:   0%|          | 0/306 [00:00<?, ?it/s]


Epoch 9:
  损失: 0.5234
  标签准确率: 0.831 (每个标签)
  严格准确率: 0.189 (8标签全对)
  验证集Macro F1: 0.3248


Training:   0%|          | 0/306 [00:00<?, ?it/s]


Epoch 10:
  损失: 0.5022
  标签准确率: 0.838 (每个标签)
  严格准确率: 0.229 (8标签全对)
  验证集Macro F1: 0.3348


Training:   0%|          | 0/306 [00:00<?, ?it/s]